In [2]:
from __future__ import annotations
from dataclasses import dataclass, field
from decimal import Decimal
from enum import Enum
from typing import Optional

In [3]:
@dataclass(frozen=True)
class Dinheiro:
    valor: Decimal

    def __post_init__(self):
        if self.valor < 0:
            raise ValueError(f"Valor monetário não pode ser negativo: {self.valor}")

    def __add__(self, outro: Dinheiro) -> Dinheiro:
        return Dinheiro(self.valor + outro.valor)

    def __sub__(self, outro: Dinheiro) -> Dinheiro:
        resultado = self.valor - outro.valor
        return Dinheiro(max(resultado, Decimal("0")))

    def __mul__(self, fator: Decimal) -> Dinheiro:
        return Dinheiro((self.valor * fator).quantize(Decimal("0.01")))

    def __gt__(self, outro: Dinheiro) -> bool:
        return self.valor > outro.valor

    def __repr__(self):
        return f"R$ {self.valor:.2f}"

`__add__`— define o operador + entre dois Dinheiro. Retorna um novo objeto (imutabilidade):

In [4]:
d1 = Dinheiro(Decimal("10"))
d2 = Dinheiro(Decimal("5"))
d3 = d1 + d2  # Dinheiro(Decimal("15")) — novo objeto

In [10]:
@dataclass(frozen=True)
class Percentual:
    valor: Decimal  # 0 a 100

    def __post_init__(self):
        if not (Decimal("0") <= self.valor <= Decimal("100")):
            raise ValueError(f"Percentual inválido: {self.valor}")

    def aplicar_sobre(self, dinheiro: Dinheiro) -> Dinheiro:
        return dinheiro * (self.valor / Decimal("100"))

Encapsula a regra de que percentual só existe entre 0 e 100. O método aplicar_sobre converte o percentual em fator e usa o `__mul__` de Dinheiro:

In [11]:
p = Percentual(Decimal("10"))
p.aplicar_sobre(Dinheiro(Decimal("129.30")))  # R$ 12.93

R$ 12.93

Sem essa classe, você espalharia essa lógica de divisão por 100 em vários lugares.

`CodigoCupom`

In [8]:
@dataclass(frozen=True)
class CodigoCupom:
    valor: str

    def __post_init__(self):
        if not self.valor or len(self.valor) < 4:
            raise ValueError("Código de cupom deve ter ao menos 4 caracteres")
        object.__setattr__(self, "valor", self.valor.upper())

`object.__setattr__`— como a dataclass é `frozen=True`, normalmente não dá para modificar atributos nem dentro do `__post_init__`. A única forma de fazer isso é chamar o `__setattr__` do object diretamente, "bypassando" a proteção do frozen. Isso é necessário para normalizar o código para maiúsculo:

In [9]:
CodigoCupom("desconto10").valor  # "DESCONTO10"
CodigoCupom("AB")                # ValueError — menos de 4 chars

ValueError: Código de cupom deve ter ao menos 4 caracteres

`Quantidade`

In [12]:
@dataclass(frozen=True)
class Quantidade:
    valor: int

    def __post_init__(self):
        if self.valor < 0:
            raise ValueError("Quantidade não pode ser negativa")

    def subtrair(self, outra: Quantidade) -> Quantidade:
        if outra.valor > self.valor:
            raise ValueError("Estoque insuficiente")
        return Quantidade(self.valor - outra.valor)

    def esta_disponivel(self) -> bool:
        return self.valor > 0

Encapsula um `int` com regras de negócio. `subtrair` protege contra estoque negativo — a regra fica dentro do objeto, não espalhada em `ifs` pelo código:

In [13]:
Quantidade(5).subtrair(Quantidade(3))   # Quantidade(2)
Quantidade(2).subtrair(Quantidade(5))   # ValueError: Estoque insuficiente
Quantidade(0).esta_disponivel()         # False

ValueError: Estoque insuficiente

Entidades de Domínio

Diferente dos Value Objects, entidades têm identidade — dois produtos com o mesmo nome são o mesmo produto.

Produto

In [14]:
@dataclass(frozen=True)
class Produto:
    nome: str
    preco: Dinheiro

    def __post_init__(self):
        if not self.nome.strip():
            raise ValueError("Produto precisa ter um nome")

Simples e imutável. frozen=True faz com que o @dataclass gere `__hash__`automaticamente, o que permite usar Produto como chave de dicionário (importante para o Estoque):

In [16]:
cafe = Produto("Café", Dinheiro(Decimal("32.90")))
inventario = {cafe: Quantidade(10)}  # funciona porque Produto é hashable

Estoque

In [17]:
class Estoque:
    def __init__(self):
        self._inventario: dict[Produto, Quantidade] = {}

    def adicionar(self, produto: Produto, quantidade: Quantidade) -> None:
        atual = self._inventario.get(produto, Quantidade(0))
        self._inventario[produto] = Quantidade(atual.valor + quantidade.valor)

    def reservar(self, produto: Produto, quantidade: Quantidade) -> None:
        disponivel = self._inventario.get(produto, Quantidade(0))
        self._inventario[produto] = disponivel.subtrair(quantidade)

    def disponivel_para(self, produto: Produto, quantidade: Quantidade) -> bool:
        atual = self._inventario.get(produto, Quantidade(0))
        return atual.valor >= quantidade.valor

    def quantidade_de(self, produto: Produto) -> Quantidade:
        return self._inventario.get(produto, Quantidade(0))

O dicionário _inventario usa Produto como chave e Quantidade como valor.
adicionar — usa .get(produto, Quantidade(0)) para não quebrar se o produto ainda não existe. Soma a nova quantidade com a atual:

In [19]:
estoque = Estoque()
estoque.adicionar(cafe, Quantidade(10))
estoque.adicionar(cafe, Quantidade(5))   # agora tem 15

In [21]:
estoque.quantidade_de(cafe)

Quantidade(valor=15)

reservar — desconta do estoque. Delega a regra de "não pode ficar negativo" para Quantidade.subtrair:

In [22]:
estoque.reservar(cafe, Quantidade(3))  # estoque vai de 15 para 12

disponivel_para — verificação antes de adicionar ao carrinho:

In [23]:
estoque.disponivel_para(cafe, Quantidade(100))  # False

False

Cupons

TipoCupom

In [24]:
class TipoCupom(Enum):
    PERCENTUAL = "percentual"
    VALOR_FIXO = "valor_fixo"

Enum é usado para representar um conjunto fixo de opções. Evita strings mágicas espalhadas pelo código:

In [ ]:
# ❌ sem Enum — frágil, erro de digitação não é detectado
if tipo == "percentual":

# ✅ com Enum — seguro, o IDE autocompleta
if tipo == TipoCupom.PERCENTUAL:

Cupom

In [26]:
@dataclass(frozen=True)
class Cupom:
    codigo: CodigoCupom
    tipo: TipoCupom
    desconto: Decimal

    def calcular_desconto(self, subtotal: Dinheiro) -> Dinheiro:
        if self.tipo == TipoCupom.PERCENTUAL:
            return Percentual(self.desconto).aplicar_sobre(subtotal)
        return Dinheiro(min(self.desconto, subtotal.valor))

O método calcular_desconto tem dois caminhos — percentual ou valor fixo. No caso de valor fixo, min(desconto, subtotal) garante que o desconto nunca ultrapassa o valor total:

In [ ]:
cupom_percent = Cupom(CodigoCupom("DESC10"), TipoCupom.PERCENTUAL, Decimal("10"))
cupom_percent.calcular_desconto(Dinheiro(Decimal("129.30")))  # R$ 12.93

cupom_fixo = Cupom(CodigoCupom("FRETE20"), TipoCupom.VALOR_FIXO, Decimal("20"))
cupom_fixo.calcular_desconto(Dinheiro(Decimal("129.30")))     # R$ 20.00
cupom_fixo.calcular_desconto(Dinheiro(Decimal("15.00")))      # R$ 15.00, não R$ 20

RepositorioDeCupons

In [ ]:
class RepositorioDeCupons:
    def __init__(self):
        self._cupons: dict[str, Cupom] = {}

    def registrar(self, cupom: Cupom) -> None:
        self._cupons[cupom.codigo.valor] = cupom

    def buscar(self, codigo: CodigoCupom) -> Optional[Cupom]:
        return self._cupons.get(codigo.valor)

Optional[Cupom] significa que pode retornar Cupom ou None. O .get() do dicionário já retorna None se a chave não existe, o que é perfeito aqui. Em produção, isso seria uma consulta ao banco de dados.

Carrinho

ItemDoCarrinho

In [27]:
@dataclass
class ItemDoCarrinho:
    produto: Produto
    quantidade: Quantidade

    def subtotal(self) -> Dinheiro:
        return self.produto.preco * Decimal(self.quantidade.valor)

Note que não é frozen=True — a quantidade pode mudar (quando você adiciona mais unidades do mesmo produto). O subtotal usa o `__mul__` de Dinheiro:

In [28]:
item = ItemDoCarrinho(cafe, Quantidade(2))
item.subtotal()  # R$ 65.80

R$ 65.80

ItensDoCarrinho — Coleção de Primeira Classe

In [30]:
class ItensDoCarrinho:
    def __init__(self):
        self._itens: list[ItemDoCarrinho] = []

    def adicionar(self, item: ItemDoCarrinho) -> None:
        existente = self._buscar_produto(item.produto)
        if existente:
            existente.quantidade = Quantidade(
                existente.quantidade.valor + item.quantidade.valor
            )
            return
        self._itens.append(item)

    def remover(self, produto: Produto) -> None:
        self._itens = [i for i in self._itens if i.produto != produto]

    def subtotal(self) -> Dinheiro:
        return sum(
            (i.subtotal() for i in self._itens),
            Dinheiro(Decimal("0"))
        )

    def esta_vazio(self) -> bool:
        return len(self._itens) == 0

    def _buscar_produto(self, produto: Produto) -> Optional[ItemDoCarrinho]:
        for item in self._itens:
            if item.produto == produto:
                return item
        return None

    def __iter__(self):
        return iter(self._itens)

adicionar — verifica se o produto já está no carrinho. Se sim, incrementa a quantidade em vez de duplicar a linha:

In [ ]:
carrinho.adicionar_produto(cafe, Quantidade(1))
carrinho.adicionar_produto(cafe, Quantidade(2))  # vira 3, não duas linhas de café

remover — list comprehension que reconstrói a lista excluindo o produto:

In [ ]:
self._itens = [i for i in self._itens if i.produto != produto]

subtotal — usa sum com valor inicial Dinheiro(Decimal("0")) porque sum começa com 0 inteiro por padrão, e 0 + Dinheiro(...) quebraria sem o `__radd__`. O valor inicial força o tipo correto:

In [ ]:
# O que sum faz internamente:
# Dinheiro("0") + item1.subtotal() + item2.subtotal() + ...

`_buscar_produto` — prefixo _ indica método privado. Retorna o item ou None:
`__iter__` — permite usar for item in itens_do_carrinho sem expor a lista diretamente.

StatusCarrinho

In [37]:
class StatusCarrinho(Enum):
    ABERTO = "aberto"
    FINALIZADO = "finalizado"
    CANCELADO = "cancelado"

Representa as transições de estado possíveis. O carrinho só pode ir de ABERTO para FINALIZADO ou CANCELADO — nunca voltar.

Carrinho — o Orquestrador

In [38]:
class Carrinho:
    def __init__(self, estoque: Estoque, repositorio_cupons: RepositorioDeCupons):
        self._itens = ItensDoCarrinho()
        self._estoque = estoque
        self._repositorio_cupons = repositorio_cupons
        self._cupom: Optional[Cupom] = None
        self._status = StatusCarrinho.ABERTO

O Carrinho recebe suas dependências por injeção — ele não cria o Estoque nem o RepositorioDeCupons, recebe prontos. Isso facilita testes e troca de implementações.

In [ ]:
def adicionar_produto(self, produto: Produto, quantidade: Quantidade) -> None:
    self._garantir_aberto()
    self._garantir_estoque(produto, quantidade)
    self._itens.adicionar(ItemDoCarrinho(produto, quantidade))

Cada método público segue o mesmo padrão: valida → executa. As validações são extraídas em métodos `_garantir_*` para manter 1 nível de indentação.

In [39]:
def aplicar_cupom(self, codigo: CodigoCupom) -> None:
    self._garantir_aberto()
    cupom = self._repositorio_cupons.buscar(codigo)
    if not cupom:
        raise ValueError(f"Cupom '{codigo.valor}' não encontrado")
    self._cupom = cupom

In [ ]:
def finalizar(self) -> Resumo:
    self._garantir_aberto()
    self._garantir_nao_vazio()
    self._reservar_estoque()
    self._status = StatusCarrinho.FINALIZADO
    return self._gerar_resumo()

finalizar é o método mais importante — faz 4 coisas em sequência e retorna um Resumo imutável. Após isso, o carrinho não pode mais ser modificado.

In [ ]:
def total(self) -> Dinheiro:
    subtotal = self._itens.subtotal()
    if not self._cupom:
        return subtotal
    return subtotal - self._cupom.calcular_desconto(subtotal)

Sem else — retorna cedo se não há cupom, aplica o desconto se há.
Métodos privados de validação:

In [41]:
def _garantir_aberto(self) -> None:
    if self._status != StatusCarrinho.ABERTO:
        raise ValueError(f"Carrinho está {self._status.value}, não pode ser modificado")

def _garantir_nao_vazio(self) -> None:
    if self._itens.esta_vazio():
        raise ValueError("Não é possível finalizar um carrinho vazio")

def _garantir_estoque(self, produto: Produto, quantidade: Quantidade) -> None:
    if not self._estoque.disponivel_para(produto, quantidade):
        disponivel = self._estoque.quantidade_de(produto)
        raise ValueError(
            f"Estoque insuficiente para '{produto.nome}': "
            f"pedido {quantidade.valor}, disponível {disponivel.valor}"
        )

def _reservar_estoque(self) -> None:
    for item in self._itens:
        self._estoque.reservar(item.produto, item.quantidade)

Cada um tem uma única responsabilidade e 0 ou 1 nível de indentação.

In [ ]:
def _gerar_resumo(self) -> Resumo:
    subtotal = self._itens.subtotal()
    desconto = self._cupom.calcular_desconto(subtotal) if self._cupom else Dinheiro(Decimal("0"))
    return Resumo(
        itens=list(self._itens),
        subtotal=subtotal,
        desconto=desconto,
        total=subtotal - desconto,
        cupom_aplicado=self._cupom,
    )

Expressão ternária x if condicao else y — evita o else em bloco e mantém o código em uma linha.

Resumo — Snapshot Imutável

In [42]:
@dataclass(frozen=True)
class Resumo:
    itens: list
    subtotal: Dinheiro
    desconto: Dinheiro
    total: Dinheiro
    cupom_aplicado: Optional[Cupom]

    def imprimir(self) -> None:
        print("\n========== RESUMO DO PEDIDO ==========")
        for item in self.itens:
            print(f"  {item.produto.nome:20s} x{item.quantidade.valor}  {item.subtotal()}")
        print("--------------------------------------")
        print(f"  Subtotal:   {self.subtotal}")
        if self.cupom_aplicado:
            print(f"  Cupom:      -{self.desconto}  ({self.cupom_aplicado.codigo.valor})")
        print(f"  TOTAL:      {self.total}")
        print("======================================\n")

Resumo é uma fotografia do pedido no momento da finalização — imutável por design. `f"  {item.produto.nome:20s}" — o :20s` formata a string com 20 caracteres de largura, alinhando os valores na impressão.

Fluxo completo comentado

In [ ]:
# 1. Infraestrutura criada uma vez
estoque = Estoque()
cupons = RepositorioDeCupons()

# 2. Produtos são value objects — imutáveis
cafe = Produto("Café Especial 250g", Dinheiro(Decimal("32.90")))

# 3. Estoque populado
estoque.adicionar(cafe, Quantidade(10))

# 4. Cupom registrado
cupons.registrar(Cupom(CodigoCupom("DESCONTO10"), TipoCupom.PERCENTUAL, Decimal("10")))

# 5. Carrinho criado com dependências injetadas
carrinho = Carrinho(estoque, cupons)

# 6. Operações — cada uma valida antes de executar
carrinho.adicionar_produto(cafe, Quantidade(2))
carrinho.aplicar_cupom(CodigoCupom("desconto10"))  # normalizado para DESCONTO10

# 7. Finalização — reserva estoque e gera snapshot
resumo = carrinho.finalizar()

# 8. Qualquer operação após finalizar lança erro
carrinho.adicionar_produto(cafe, Quantidade(1))  # ValueError!
```

---

## Mapa mental das dependências
```
Carrinho
  ├── usa → ItensDoCarrinho
  │             └── contém → ItemDoCarrinho[]
  │                             ├── tem → Produto
  │                             │           └── tem → Dinheiro
  │                             └── tem → Quantidade
  ├── usa → Estoque
  │             └── guarda → {Produto: Quantidade}
  ├── usa → RepositorioDeCupons
  │             └── guarda → {str: Cupom}
  │                               └── tem → CodigoCupom
  │                                         TipoCupom
  │                                         Percentual (quando aplica)
  └── produz → Resumo (imutável)